# 02_underwriting_cfe
Train multiple underwriting-style risk models on KHPS actionable features and
covariates, then generate counterfactual-explanation (CFE) recourse for the
declined group with DiCE. Multiple models (gradient-boosted trees + logistic)
are used deliberately so that later feasibility findings are shown to be
data-governed rather than an artefact of a single classifier (defence against the
'synthetic-label circularity' critique). The target is baseline HTN status used
as an underwriting-risk proxy; recourse is prescribed only over actionable
features. Outputs: fitted models, a declined-group recourse table, and a
model-agreement figure.

In [1]:
%run 00_config.ipynb

PROJ_DIR: /home/claude/recourse_khp
1y pairs: [(2019, 2020), (2020, 2021), (2021, 2022), (2022, 2023), (2023, 2024)]
2y pairs: [(2019, 2021), (2020, 2022), (2021, 2023), (2022, 2024)]
registry loaded
helpers loaded
00_config ready


In [2]:
import joblib
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from sklearn.ensemble import GradientBoostingClassifier
import dice_ml
RNG=42; np.random.seed(RNG)

In [3]:
# --- Assemble the modelling frame from the long panel (baseline wave per person) ---
panel=pd.read_parquet(os.path.join(DATA_DIR,"panel_long.parquet"))
# Use the earliest observation per person that has complete actionable+covariate data
ACT=["BMI","PA_REG","PA_WALK","ALC_FREQ"]          # primary actionable set
COV=["age","SEX","EDU","H_INC_TOT"]                # non-actionable covariates in model
feat=ACT+COV
d=panel.dropna(subset=feat+["HTN_dx"]).copy()
d["SEX"]=(d["SEX"]=="M").astype(int)              # encode
for col in ["PA_REG","EDU"]:
    d[col]=pd.to_numeric(d[col], errors="coerce")
d=d.sort_values(["year"]).groupby(KEY, as_index=False).first()   # one row per person
X=d[feat].astype(float); y=d["HTN_dx"].astype(int)  # y=1 -> higher underwriting risk
print("modelling frame:", X.shape, "| positive(HTN) rate:", round(y.mean(),3))

modelling frame: (15389, 8) | positive(HTN) rate: 0.298


In [4]:
# --- Train/test split ---
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.3,stratify=y,random_state=RNG)

# Model A: gradient-boosted trees = PRIMARY underwriting model (sklearn GBM).
# sklearn GBM is the DiCE-facing model (stable dtype handling under genetic search).
mA=GradientBoostingClassifier(n_estimators=300,max_depth=3,learning_rate=0.05,
                              subsample=0.9,random_state=RNG)
mA.fit(Xtr,ytr)

# Second tree model (XGBoost) and Model B (L2 logistic) exist ONLY for the
# model-agreement check that defends against single-classifier circularity.
mXGB=XGBClassifier(n_estimators=300,max_depth=3,learning_rate=0.05,
                   subsample=0.9,colsample_bytree=0.9,eval_metric="logloss",
                   random_state=RNG).fit(Xtr,ytr)
mB=Pipeline([("sc",StandardScaler()),
             ("lr",LogisticRegression(max_iter=2000,C=1.0,random_state=RNG))]).fit(Xtr,ytr)

for nm,m in [("GBM(primary)",mA),("XGBoost",mXGB),("Logistic",mB)]:
    auc=roc_auc_score(yte, m.predict_proba(Xte)[:,1])
    print(f"{nm:14s} test AUC = {auc:.3f}")
joblib.dump({"mA":mA,"mXGB":mXGB,"mB":mB,"feat":feat,"ACT":ACT,"COV":COV},
            os.path.join(DATA_DIR,"uw_models.joblib"))

GBM(primary)   test AUC = 0.835
XGBoost        test AUC = 0.833
Logistic       test AUC = 0.835


['/home/claude/recourse_khp/data/uw_models.joblib']

In [5]:
# --- Define 'declined' = predicted high risk by the primary model (top-risk band) ---
# Underwriting analogue: applicants whose predicted risk exceeds a threshold are declined.
p_riskA=mA.predict_proba(X)[:,1]
thr=np.quantile(p_riskA, 0.70)          # decline the top ~30% risk as illustration
d["p_risk_A"]=p_riskA
d["declined_A"]=(p_riskA>=thr).astype(int)
print("decline threshold (risk prob):",round(float(thr),3),
      "| declined share:",round(d["declined_A"].mean(),3))

decline threshold (risk prob): 0.472 | declined share: 0.3


In [6]:
# --- DiCE recourse for the declined group over ACTIONABLE features only ---
data_dice=dice_ml.Data(dataframe=pd.concat([X, y.rename("HTN")],axis=1),
                       continuous_features=[c for c in feat if c not in ["SEX"]],
                       outcome_name="HTN")
model_dice=dice_ml.Model(model=mA, backend="sklearn", model_type="classifier")
exp=dice_ml.Dice(data_dice, model_dice, method="genetic")   # sparser, proximity-aware CFEs

declined=d[d["declined_A"]==1].copy()
# sample for tractable runtime; keep a fixed seed sample
n_cf=min(600, len(declined))
q=declined.sample(n_cf, random_state=RNG)[feat].astype(float).reset_index(drop=True)
print("generating CFEs for", len(q), "declined cases (actionable-only)...")

# Risk-reducing recourse should only *lower* BMI/drinking and *raise* activity.
# Encode that as one-sided permitted ranges anchored at each person's baseline so
# the CF cannot prescribe an implausible direction (e.g. drink more to lower risk).
def make_cf(row_df):
    r=row_df.iloc[0]
    pr={"BMI":[16.0, float(r["BMI"])],           # BMI may only go down
        "PA_WALK":[float(r["PA_WALK"]),7.0],      # walking may only go up
        "ALC_FREQ":[0.0, float(r["ALC_FREQ"])],   # drinking may only go down
        "PA_REG":[float(r["PA_REG"]),1.0]}        # regular exercise may only go up (0->1)
    # guard against degenerate ranges (lo==hi) which DiCE rejects
    for k,(lo,hi) in list(pr.items()):
        if hi<=lo: pr[k]=[lo,hi+1e-6] if k in ("PA_WALK","PA_REG") else [lo-1e-6,hi]
    return exp.generate_counterfactuals(
        row_df, total_CFs=1, desired_class=0,
        features_to_vary=ACT, permitted_range=pr,
        proximity_weight=1.5, sparsity_weight=1.0)

cf_list=[]
for i in range(len(q)):
    try:
        one=make_cf(q.iloc[[i]])
        cf_list.append(one.cf_examples_list[0])
    except Exception:
        cf_list.append(None)
print("done. valid CFs:", sum(x is not None for x in cf_list),"/",len(q))

generating CFEs for 600 declined cases (actionable-only)...


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00,  6.72it/s]

100%|██████████| 1/1 [00:00<00:00,  6.66it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

...

100%|██████████| 1/1 [00:00<00:00,  7.13it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00,  7.56it/s]

100%|██████████| 1/1 [00:00<00:00,  7.48it/s]

done. valid CFs: 540 / 600


In [7]:
# --- Extract recourse direction (delta) per case ---
recs=[]
for i in range(len(q)):
    orig=q.iloc[i]
    ex=cf_list[i]
    if ex is None: continue
    cfdf=ex.final_cfs_df
    if cfdf is None or len(cfdf)==0:
        continue
    new=cfdf.iloc[0]
    row={"idx":i}
    for a in ACT:
        row[f"{a}_0"]=float(orig[a]); row[f"{a}_cf"]=float(new[a])
        row[f"d_{a}"]=float(new[a])-float(orig[a])
    recs.append(row)
rec=pd.DataFrame(recs)
# Direction summary: BMI should go down, PA up, ALC down for risk reduction
print("recourse cases with a valid CF:", len(rec))
for a in ACT:
    print(f"  mean prescribed delta {a}: {rec[f'd_{a}'].mean():+.3f}")
rec.to_parquet(os.path.join(DATA_DIR,"recourse_declined.parquet"), index=False)
savetable(rec.describe().round(3), "t02_recourse_delta_summary")

recourse cases with a valid CF: 540
  mean prescribed delta BMI: -4.703
  mean prescribed delta PA_REG: +0.000
  mean prescribed delta PA_WALK: -0.007
  mean prescribed delta ALC_FREQ: -0.270
saved: t02_recourse_delta_summary.csv


,idx,BMI_0,BMI_cf,d_BMI,PA_REG_0,PA_REG_cf,d_PA_REG,PA_WALK_0,PA_WALK_cf,d_PA_WALK,ALC_FREQ_0,ALC_FREQ_cf,d_ALC_FREQ
count,540.000,540.000,540.000,540.000,540.000,540.000,540.0,540.000,540.000,540.000,540.000,540.000,540.000
mean,298.546,24.965,20.261,-4.703,0.561,0.561,0.0,5.741,5.734,-0.007,1.930,1.660,-0.270
std,173.359,2.882,2.764,3.251,0.497,0.497,0.0,2.067,2.051,0.113,2.119,1.831,0.859
min,0.000,17.857,16.000,-16.873,0.000,0.000,0.0,1.000,1.000,-1.000,0.000,0.000,-6.000
25%,148.750,23.191,18.496,-6.614,0.000,0.000,0.0,4.000,4.000,0.000,0.000,0.000,0.000
50%,297.500,24.609,20.412,-4.108,1.000,1.000,0.0,7.000,7.000,0.000,1.000,1.000,0.000
75%,445.250,26.667,21.967,-2.338,1.000,1.000,0.0,7.000,7.000,0.000,4.000,3.000,0.000
max,599.000,37.949,30.469,0.000,1.000,1.000,0.0,8.000,8.000,1.000,7.000,7.000,0.000


In [8]:
# --- Model-agreement figure: do the underwriting models rank risk similarly? ---
from scipy.stats import spearmanr
pGBM=mA.predict_proba(X)[:,1]; pXGB=mXGB.predict_proba(X)[:,1]; pLR=mB.predict_proba(X)[:,1]
rho_x,_=spearmanr(pGBM,pXGB); rho_l,_=spearmanr(pGBM,pLR)
fig,axes=plt.subplots(1,2,figsize=(8.4,4.2))
for ax,(pp,lab,rho) in zip(axes,[(pXGB,"XGBoost",rho_x),(pLR,"Logistic",rho_l)]):
    ax.scatter(pGBM,pp,s=6,alpha=0.25,color="#333333",edgecolor="none")
    ax.plot([0,1],[0,1],color="#000000",lw=0.8,ls="--")
    ax.set_xlabel("Predicted risk (GBM)"); ax.set_ylabel(f"Predicted risk ({lab})")
    ax.set_xlim(0,1); ax.set_ylim(0,1)
    ax.text(0.05,0.92,f"Spearman rho = {rho:.2f}",transform=ax.transAxes)
savefig(fig,"f02_model_agreement"); plt.close(fig)
print("risk-rank agreement  GBM~XGB:",round(rho_x,3)," GBM~LR:",round(rho_l,3))

saved: f02_model_agreement.png / f02_model_agreement.pdf
risk-rank agreement  GBM~XGB: 0.995  GBM~LR: 0.978
